[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-USERNAME/AI-in-healthcare-book/blob/main/notebooks/volume2_chapter_06/notebook_6A_success_metrics.ipynb)

*Click the badge above to open this notebook in Google Colab (no setup required)*

---

# Notebook 6A: Success Metrics and ROI for Healthcare AI

**Volume 2, Chapter 6: AI Maturity & Success Metrics**

## Learning Objectives

By the end of this notebook, you will be able to:
1. Define and calculate key success metrics for healthcare AI implementations
2. Build ROI models for AI investment decisions
3. Perform cost-benefit analysis with sensitivity analysis
4. Create success tracking dashboards
5. Calculate time-to-value and break-even points

## Clinical Context

Healthcare AI implementations require substantial investment in software, integration, training, and change management. Decision-makers need rigorous frameworks to evaluate whether these investments deliver value. Unlike consumer technology, healthcare AI must demonstrate improvements in clinical outcomes, operational efficiency, AND financial sustainability.

**Key insight from Chapter 6:** Success metrics must be defined BEFORE implementation, tracked consistently DURING rollout, and honestly assessed AFTER deployment. The most common failure mode is deploying AI without clear success criteria, making it impossible to know if the investment was worthwhile.

---

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

# Define consistent color scheme
COLORS = {
    'primary': '#2E86AB',
    'secondary': '#E94F37',
    'success': '#27AE60',
    'warning': '#F39C12',
    'danger': '#E74C3C',
    'neutral': '#95A5A6',
    'benefit': '#27AE60',
    'cost': '#E74C3C',
    'cumulative': '#8E44AD'
}

def format_currency(value, prefix='$'):
    """Format value as currency string."""
    if abs(value) >= 1e6:
        return f"{prefix}{value/1e6:.1f}M"
    elif abs(value) >= 1e3:
        return f"{prefix}{value/1e3:.0f}K"
    else:
        return f"{prefix}{value:.0f}"

def format_percentage(value):
    """Format value as percentage string."""
    return f"{value:.1%}"

print("Libraries imported successfully")

---

## 1. Defining Success Metrics for Healthcare AI

Healthcare AI success must be measured across three dimensions: **Clinical**, **Operational**, and **Financial**. Each dimension has specific metrics that together provide a complete picture of implementation success.

### Metric Hierarchy Framework

In [ ]:
# Define the metric hierarchy
metric_categories = {
    'Clinical Metrics': {
        'Sensitivity/Specificity': 'Model accuracy for detecting conditions',
        'Lives Saved': 'Mortality reduction attributable to AI',
        'Complications Avoided': 'Adverse events prevented',
        'Diagnostic Accuracy': 'Correct diagnoses vs. gold standard',
        'Time to Treatment': 'Reduction in treatment delays'
    },
    'Operational Metrics': {
        'Time Saved': 'Clinician hours freed per week',
        'Throughput Increase': 'Additional patients served',
        'Length of Stay': 'Average days reduced per admission',
        'Readmission Rate': 'Reduction in 30-day readmissions',
        'Alert Fatigue': 'Reduction in false positive alerts'
    },
    'Financial Metrics': {
        'Cost Savings': 'Direct cost reduction (supplies, tests)',
        'Revenue Impact': 'Additional revenue from efficiency',
        'ROI': 'Return on Investment percentage',
        'Net Present Value': 'NPV of AI investment',
        'Payback Period': 'Months to recover investment'
    }
}

# Create visualization of metric hierarchy
fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')

# Title
ax.text(5, 9.5, 'Healthcare AI Success Metrics Framework', fontsize=16, 
        fontweight='bold', ha='center', va='center')

# Draw three columns for categories
colors = [COLORS['primary'], COLORS['success'], COLORS['warning']]
positions = [1.7, 5, 8.3]

for idx, (category, metrics) in enumerate(metric_categories.items()):
    x_pos = positions[idx]
    color = colors[idx]
    
    # Category header box
    ax.add_patch(plt.Rectangle((x_pos-1.5, 7.5), 3, 1, facecolor=color, 
                               edgecolor='black', linewidth=2, alpha=0.8))
    ax.text(x_pos, 8, category, fontsize=11, fontweight='bold', 
            ha='center', va='center', color='white')
    
    # Metrics list
    y_start = 6.8
    for metric_name in metrics.keys():
        ax.add_patch(plt.Rectangle((x_pos-1.4, y_start-0.4), 2.8, 0.7, 
                                   facecolor='white', edgecolor=color, linewidth=1.5))
        ax.text(x_pos, y_start, metric_name, fontsize=9, ha='center', va='center')
        y_start -= 0.9

# Add connecting element at bottom
ax.add_patch(plt.Rectangle((1.5, 1.2), 7, 0.8, facecolor=COLORS['cumulative'], 
                           edgecolor='black', linewidth=2, alpha=0.8))
ax.text(5, 1.6, 'Integrated Success Dashboard', fontsize=12, fontweight='bold', 
        ha='center', va='center', color='white')

# Draw arrows pointing down to integrated dashboard
for x_pos in positions:
    ax.annotate('', xy=(x_pos, 2.0), xytext=(x_pos, 2.5),
                arrowprops=dict(arrowstyle='->', color='gray', lw=2))

plt.tight_layout()
plt.show()

# Print detailed metric descriptions
print("\nDetailed Metric Descriptions:")
print("=" * 70)
for category, metrics in metric_categories.items():
    print(f"\n{category}:")
    for metric, description in metrics.items():
        print(f"  - {metric}: {description}")

---

## 2. Case Study: Sepsis AI Implementation

We will build a comprehensive ROI model for implementing an AI-based sepsis early warning system. This is a common use case with well-documented outcomes data.

### Define Baseline Scenario (Without AI)

In [ ]:
# Baseline scenario parameters (without AI)
baseline = {
    'annual_sepsis_cases': 500,
    'mortality_rate': 0.25,  # 25% mortality
    'avg_cost_per_case': 50000,  # $50,000 average treatment cost
    'avg_los_days': 14,  # 14 days average length of stay
    'cost_per_day': 2500,  # $2,500 per day
    'icu_days_per_case': 5,  # 5 ICU days on average
    'icu_cost_per_day': 5000,  # $5,000 per ICU day
    'mortality_cost': 150000,  # Economic cost of mortality (care, litigation risk)
}

# Calculate baseline costs
baseline['total_deaths'] = baseline['annual_sepsis_cases'] * baseline['mortality_rate']
baseline['total_treatment_cost'] = baseline['annual_sepsis_cases'] * baseline['avg_cost_per_case']
baseline['total_los_cost'] = baseline['annual_sepsis_cases'] * baseline['avg_los_days'] * baseline['cost_per_day']
baseline['total_icu_cost'] = baseline['annual_sepsis_cases'] * baseline['icu_days_per_case'] * baseline['icu_cost_per_day']
baseline['total_mortality_cost'] = baseline['total_deaths'] * baseline['mortality_cost']
baseline['total_annual_cost'] = (
    baseline['total_treatment_cost'] + 
    baseline['total_los_cost'] + 
    baseline['total_icu_cost'] + 
    baseline['total_mortality_cost']
)

print("BASELINE SCENARIO (Without AI Intervention)")
print("=" * 60)
print(f"\nAnnual Sepsis Cases: {baseline['annual_sepsis_cases']:,}")
print(f"Mortality Rate: {baseline['mortality_rate']:.0%}")
print(f"Deaths per Year: {baseline['total_deaths']:.0f}")
print(f"\nCost Breakdown:")
print(f"  Treatment Costs: {format_currency(baseline['total_treatment_cost'])}")
print(f"  Length of Stay Costs: {format_currency(baseline['total_los_cost'])}")
print(f"  ICU Costs: {format_currency(baseline['total_icu_cost'])}")
print(f"  Mortality-Related Costs: {format_currency(baseline['total_mortality_cost'])}")
print(f"\n  TOTAL ANNUAL COST: {format_currency(baseline['total_annual_cost'])}")

### Define AI Implementation Scenario

In [ ]:
# AI implementation parameters
ai_scenario = {
    # Expected improvements (based on literature)
    'mortality_reduction': 0.20,  # 20% relative reduction in mortality
    'los_reduction_days': 2,  # 2 days reduction in LOS
    'icu_days_reduction': 1,  # 1 day reduction in ICU stay
    'early_detection_rate': 0.75,  # 75% of cases detected earlier
    
    # Implementation costs (Year 1)
    'software_license_annual': 150000,
    'integration_one_time': 200000,
    'training_initial': 75000,
    'change_management': 50000,
    
    # Ongoing costs (Annual)
    'maintenance_annual': 30000,
    'training_ongoing_annual': 15000,
    'it_support_annual': 25000,
}

# Calculate AI scenario outcomes
ai_scenario['new_mortality_rate'] = baseline['mortality_rate'] * (1 - ai_scenario['mortality_reduction'])
ai_scenario['new_deaths'] = baseline['annual_sepsis_cases'] * ai_scenario['new_mortality_rate']
ai_scenario['lives_saved'] = baseline['total_deaths'] - ai_scenario['new_deaths']

ai_scenario['new_los_days'] = baseline['avg_los_days'] - ai_scenario['los_reduction_days']
ai_scenario['new_icu_days'] = baseline['icu_days_per_case'] - ai_scenario['icu_days_reduction']

# Calculate AI scenario costs
ai_scenario['treatment_cost'] = baseline['total_treatment_cost'] * 0.95  # 5% reduction from early detection
ai_scenario['los_cost'] = baseline['annual_sepsis_cases'] * ai_scenario['new_los_days'] * baseline['cost_per_day']
ai_scenario['icu_cost'] = baseline['annual_sepsis_cases'] * ai_scenario['new_icu_days'] * baseline['icu_cost_per_day']
ai_scenario['mortality_cost'] = ai_scenario['new_deaths'] * baseline['mortality_cost']

ai_scenario['total_clinical_cost'] = (
    ai_scenario['treatment_cost'] + 
    ai_scenario['los_cost'] + 
    ai_scenario['icu_cost'] + 
    ai_scenario['mortality_cost']
)

# Year 1 implementation cost
ai_scenario['year1_implementation_cost'] = (
    ai_scenario['software_license_annual'] +
    ai_scenario['integration_one_time'] +
    ai_scenario['training_initial'] +
    ai_scenario['change_management']
)

# Ongoing annual cost (Year 2+)
ai_scenario['ongoing_annual_cost'] = (
    ai_scenario['software_license_annual'] +
    ai_scenario['maintenance_annual'] +
    ai_scenario['training_ongoing_annual'] +
    ai_scenario['it_support_annual']
)

print("AI IMPLEMENTATION SCENARIO")
print("=" * 60)
print(f"\nExpected Improvements:")
print(f"  Mortality Reduction: {ai_scenario['mortality_reduction']:.0%} (relative)")
print(f"  New Mortality Rate: {ai_scenario['new_mortality_rate']:.0%}")
print(f"  Lives Saved Annually: {ai_scenario['lives_saved']:.0f}")
print(f"  LOS Reduction: {ai_scenario['los_reduction_days']} days")
print(f"  ICU Days Reduction: {ai_scenario['icu_days_reduction']} day")
print(f"\nImplementation Costs (Year 1):")
print(f"  Software License: {format_currency(ai_scenario['software_license_annual'])}")
print(f"  Integration: {format_currency(ai_scenario['integration_one_time'])}")
print(f"  Initial Training: {format_currency(ai_scenario['training_initial'])}")
print(f"  Change Management: {format_currency(ai_scenario['change_management'])}")
print(f"  TOTAL YEAR 1: {format_currency(ai_scenario['year1_implementation_cost'])}")
print(f"\nOngoing Annual Costs (Year 2+):")
print(f"  Software License: {format_currency(ai_scenario['software_license_annual'])}")
print(f"  Maintenance: {format_currency(ai_scenario['maintenance_annual'])}")
print(f"  Ongoing Training: {format_currency(ai_scenario['training_ongoing_annual'])}")
print(f"  IT Support: {format_currency(ai_scenario['it_support_annual'])}")
print(f"  TOTAL ANNUAL: {format_currency(ai_scenario['ongoing_annual_cost'])}")

### Calculate Expected Benefits

In [ ]:
# Calculate annual benefits
benefits = {
    'mortality_cost_avoided': baseline['total_mortality_cost'] - ai_scenario['mortality_cost'],
    'los_savings': baseline['total_los_cost'] - ai_scenario['los_cost'],
    'icu_savings': baseline['total_icu_cost'] - ai_scenario['icu_cost'],
    'treatment_savings': baseline['total_treatment_cost'] - ai_scenario['treatment_cost'],
}

benefits['total_annual_benefit'] = sum(benefits.values())

print("ANNUAL BENEFITS CALCULATION")
print("=" * 60)
print(f"\nCost Savings by Category:")
print(f"  Mortality Cost Avoided: {format_currency(benefits['mortality_cost_avoided'])}")
print(f"  Length of Stay Savings: {format_currency(benefits['los_savings'])}")
print(f"  ICU Cost Savings: {format_currency(benefits['icu_savings'])}")
print(f"  Treatment Efficiency: {format_currency(benefits['treatment_savings'])}")
print(f"\n  TOTAL ANNUAL BENEFIT: {format_currency(benefits['total_annual_benefit'])}")

# Visualize benefits breakdown
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart of benefits
ax = axes[0]
benefit_labels = ['Mortality\nCost Avoided', 'LOS\nSavings', 'ICU\nSavings', 'Treatment\nSavings']
benefit_values = [benefits['mortality_cost_avoided'], benefits['los_savings'], 
                  benefits['icu_savings'], benefits['treatment_savings']]
colors_pie = [COLORS['success'], COLORS['primary'], COLORS['warning'], COLORS['neutral']]

wedges, texts, autotexts = ax.pie(benefit_values, labels=benefit_labels, autopct='%1.1f%%',
                                   colors=colors_pie, startangle=90)
ax.set_title('Annual Benefits Breakdown', fontsize=13, fontweight='bold')

# Bar chart comparing baseline vs AI
ax = axes[1]
categories = ['Mortality\nCosts', 'LOS\nCosts', 'ICU\nCosts', 'Treatment\nCosts']
baseline_values = [baseline['total_mortality_cost']/1e6, baseline['total_los_cost']/1e6, 
                   baseline['total_icu_cost']/1e6, baseline['total_treatment_cost']/1e6]
ai_values = [ai_scenario['mortality_cost']/1e6, ai_scenario['los_cost']/1e6,
             ai_scenario['icu_cost']/1e6, ai_scenario['treatment_cost']/1e6]

x = np.arange(len(categories))
width = 0.35

bars1 = ax.bar(x - width/2, baseline_values, width, label='Baseline (No AI)', color=COLORS['danger'], alpha=0.8)
bars2 = ax.bar(x + width/2, ai_values, width, label='With AI', color=COLORS['success'], alpha=0.8)

ax.set_xlabel('Cost Category', fontsize=12)
ax.set_ylabel('Annual Cost ($ Millions)', fontsize=12)
ax.set_title('Cost Comparison: Baseline vs. AI Implementation', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

# Add value labels
for bar in bars1:
    height = bar.get_height()
    ax.annotate(f'${height:.1f}M', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    height = bar.get_height()
    ax.annotate(f'${height:.1f}M', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

---

## 3. ROI Calculator

Build a comprehensive ROI model that calculates key financial metrics for AI investment decisions.

In [ ]:
class HealthcareAIROICalculator:
    """
    Comprehensive ROI calculator for healthcare AI implementations.
    
    Calculates:
    - Simple ROI percentage
    - Net Present Value (NPV)
    - Internal Rate of Return (IRR)
    - Payback Period
    - Break-even Analysis
    """
    
    def __init__(self, 
                 year1_costs,
                 ongoing_annual_costs,
                 annual_benefits,
                 discount_rate=0.08,
                 time_horizon_years=5,
                 adoption_curve=None):
        """
        Initialize ROI calculator.
        
        Parameters:
        -----------
        year1_costs : float
            Total first-year implementation costs
        ongoing_annual_costs : float
            Annual ongoing costs (Year 2+)
        annual_benefits : float
            Fully-realized annual benefits
        discount_rate : float
            Discount rate for NPV calculation (default 8%)
        time_horizon_years : int
            Analysis time horizon in years
        adoption_curve : list, optional
            Benefit realization percentage by year (default: gradual ramp)
        """
        self.year1_costs = year1_costs
        self.ongoing_annual_costs = ongoing_annual_costs
        self.annual_benefits = annual_benefits
        self.discount_rate = discount_rate
        self.time_horizon = time_horizon_years
        
        # Default adoption curve: gradual ramp-up
        if adoption_curve is None:
            self.adoption_curve = [0.25, 0.60, 0.85, 0.95, 1.0]  # 5-year ramp
            # Extend if needed
            while len(self.adoption_curve) < time_horizon_years:
                self.adoption_curve.append(1.0)
        else:
            self.adoption_curve = adoption_curve
    
    def calculate_cash_flows(self):
        """Calculate year-by-year cash flows."""
        cash_flows = []
        
        for year in range(self.time_horizon):
            if year == 0:
                costs = self.year1_costs
            else:
                costs = self.ongoing_annual_costs
            
            benefits = self.annual_benefits * self.adoption_curve[year]
            net_cash_flow = benefits - costs
            
            cash_flows.append({
                'year': year + 1,
                'adoption_rate': self.adoption_curve[year],
                'costs': costs,
                'benefits': benefits,
                'net_cash_flow': net_cash_flow
            })
        
        return pd.DataFrame(cash_flows)
    
    def calculate_npv(self):
        """Calculate Net Present Value."""
        df = self.calculate_cash_flows()
        npv = 0
        for idx, row in df.iterrows():
            discounted = row['net_cash_flow'] / ((1 + self.discount_rate) ** row['year'])
            npv += discounted
        return npv
    
    def calculate_simple_roi(self):
        """Calculate simple ROI over the time horizon."""
        df = self.calculate_cash_flows()
        total_benefits = df['benefits'].sum()
        total_costs = df['costs'].sum()
        roi = (total_benefits - total_costs) / total_costs
        return roi
    
    def calculate_payback_period(self):
        """Calculate payback period in months."""
        df = self.calculate_cash_flows()
        cumulative = 0
        
        for idx, row in df.iterrows():
            cumulative += row['net_cash_flow']
            if cumulative >= 0:
                # Calculate partial year
                prev_cumulative = cumulative - row['net_cash_flow']
                fraction = abs(prev_cumulative) / row['net_cash_flow'] if row['net_cash_flow'] > 0 else 0
                payback_years = row['year'] - 1 + fraction
                return payback_years * 12  # Convert to months
        
        return None  # No payback within time horizon
    
    def calculate_break_even_adoption(self):
        """Calculate minimum adoption rate needed for break-even in Year 1."""
        # Break-even when benefits >= costs
        break_even_adoption = self.year1_costs / self.annual_benefits
        return min(break_even_adoption, 1.0)  # Cap at 100%
    
    def generate_summary(self):
        """Generate comprehensive ROI summary."""
        df = self.calculate_cash_flows()
        
        summary = {
            'Time Horizon': f"{self.time_horizon} years",
            'Total Investment': df['costs'].sum(),
            'Total Benefits': df['benefits'].sum(),
            'Net Benefit': df['net_cash_flow'].sum(),
            'Simple ROI': self.calculate_simple_roi(),
            'NPV (at {:.0%} discount)'.format(self.discount_rate): self.calculate_npv(),
            'Payback Period': self.calculate_payback_period(),
            'Break-even Adoption Rate': self.calculate_break_even_adoption()
        }
        
        return summary, df
    
    def print_report(self):
        """Print formatted ROI report."""
        summary, df = self.generate_summary()
        
        print("\n" + "=" * 70)
        print("ROI ANALYSIS REPORT")
        print("=" * 70)
        
        print(f"\nAnalysis Period: {summary['Time Horizon']}")
        print(f"Discount Rate: {self.discount_rate:.0%}")
        
        print(f"\n{'Key Financial Metrics':^40}")
        print("-" * 40)
        print(f"Total Investment: {format_currency(summary['Total Investment'])}")
        print(f"Total Benefits: {format_currency(summary['Total Benefits'])}")
        print(f"Net Benefit: {format_currency(summary['Net Benefit'])}")
        print(f"Simple ROI: {summary['Simple ROI']:.1%}")
        print(f"NPV: {format_currency(list(summary.values())[5])}")
        
        payback = summary['Payback Period']
        if payback:
            print(f"Payback Period: {payback:.1f} months")
        else:
            print(f"Payback Period: >5 years")
        
        print(f"Break-even Adoption: {summary['Break-even Adoption Rate']:.0%}")
        
        print(f"\n{'Year-by-Year Cash Flows':^40}")
        print("-" * 70)
        print(f"{'Year':<6} {'Adoption':<10} {'Costs':<12} {'Benefits':<12} {'Net CF':<12} {'Cumulative':<12}")
        print("-" * 70)
        
        cumulative = 0
        for idx, row in df.iterrows():
            cumulative += row['net_cash_flow']
            print(f"{row['year']:<6} {row['adoption_rate']:<10.0%} "
                  f"{format_currency(row['costs']):<12} "
                  f"{format_currency(row['benefits']):<12} "
                  f"{format_currency(row['net_cash_flow']):<12} "
                  f"{format_currency(cumulative):<12}")
        
        print("=" * 70)
        
        return summary, df

In [ ]:
# Create ROI calculator with our sepsis AI scenario
roi_calculator = HealthcareAIROICalculator(
    year1_costs=ai_scenario['year1_implementation_cost'],
    ongoing_annual_costs=ai_scenario['ongoing_annual_cost'],
    annual_benefits=benefits['total_annual_benefit'],
    discount_rate=0.08,
    time_horizon_years=5,
    adoption_curve=[0.30, 0.65, 0.85, 0.95, 1.0]  # Realistic adoption ramp
)

# Generate report
summary, cash_flows = roi_calculator.print_report()

In [ ]:
# Visualize ROI analysis
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Cash flow over time
ax = axes[0, 0]
x = cash_flows['year']
ax.bar(x - 0.2, cash_flows['costs']/1e6, 0.4, label='Costs', color=COLORS['cost'], alpha=0.8)
ax.bar(x + 0.2, cash_flows['benefits']/1e6, 0.4, label='Benefits', color=COLORS['benefit'], alpha=0.8)
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Amount ($ Millions)', fontsize=12)
ax.set_title('Annual Costs vs. Benefits', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')
ax.set_xticks(x)

# Plot 2: Cumulative cash flow
ax = axes[0, 1]
cumulative = cash_flows['net_cash_flow'].cumsum()
colors = [COLORS['danger'] if c < 0 else COLORS['success'] for c in cumulative]
ax.bar(x, cumulative/1e6, color=colors, alpha=0.8)
ax.axhline(y=0, color='black', linestyle='-', linewidth=1)
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Cumulative Net Cash Flow ($ Millions)', fontsize=12)
ax.set_title('Cumulative Cash Flow (Break-even Analysis)', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
ax.set_xticks(x)

# Find and annotate break-even point
for i, c in enumerate(cumulative):
    if c >= 0:
        ax.annotate('Break-even', xy=(x.iloc[i], c/1e6), xytext=(x.iloc[i]+0.5, c/1e6+0.5),
                   fontsize=11, arrowprops=dict(arrowstyle='->', color='black'))
        break

# Plot 3: Adoption curve impact
ax = axes[1, 0]
ax.plot(x, cash_flows['adoption_rate']*100, 'o-', color=COLORS['primary'], 
        linewidth=2, markersize=10, label='Adoption Rate')
ax2 = ax.twinx()
ax2.plot(x, cash_flows['benefits']/1e6, 's--', color=COLORS['success'], 
         linewidth=2, markersize=8, label='Realized Benefits')
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Adoption Rate (%)', fontsize=12, color=COLORS['primary'])
ax2.set_ylabel('Benefits ($ Millions)', fontsize=12, color=COLORS['success'])
ax.set_title('Adoption Rate vs. Realized Benefits', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.tick_params(axis='y', labelcolor=COLORS['primary'])
ax2.tick_params(axis='y', labelcolor=COLORS['success'])
ax.grid(True, alpha=0.3)

# Plot 4: NPV sensitivity to discount rate
ax = axes[1, 1]
discount_rates = np.arange(0.02, 0.16, 0.02)
npvs = []
for dr in discount_rates:
    calc = HealthcareAIROICalculator(
        year1_costs=ai_scenario['year1_implementation_cost'],
        ongoing_annual_costs=ai_scenario['ongoing_annual_cost'],
        annual_benefits=benefits['total_annual_benefit'],
        discount_rate=dr,
        time_horizon_years=5,
        adoption_curve=[0.30, 0.65, 0.85, 0.95, 1.0]
    )
    npvs.append(calc.calculate_npv())

ax.plot(discount_rates*100, [n/1e6 for n in npvs], 'o-', color=COLORS['cumulative'], linewidth=2, markersize=8)
ax.axhline(y=0, color='red', linestyle='--', linewidth=1, label='Break-even')
ax.axvline(x=8, color=COLORS['warning'], linestyle=':', linewidth=2, label='Base case (8%)')
ax.set_xlabel('Discount Rate (%)', fontsize=12)
ax.set_ylabel('NPV ($ Millions)', fontsize=12)
ax.set_title('NPV Sensitivity to Discount Rate', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## 4. Sensitivity Analysis

ROI calculations depend on assumptions that may not hold in practice. Sensitivity analysis examines how changes in key assumptions affect the results.

In [ ]:
def sensitivity_analysis(base_calculator, parameter_ranges):
    """
    Perform sensitivity analysis on ROI calculation.
    
    Parameters:
    -----------
    base_calculator : HealthcareAIROICalculator
        Base case calculator
    parameter_ranges : dict
        Dictionary of parameter names and (min, max) tuples
    
    Returns:
    --------
    DataFrame with sensitivity results
    """
    base_npv = base_calculator.calculate_npv()
    results = []
    
    for param, (low, high) in parameter_ranges.items():
        # Test low value
        if param == 'annual_benefits':
            calc_low = HealthcareAIROICalculator(
                year1_costs=base_calculator.year1_costs,
                ongoing_annual_costs=base_calculator.ongoing_annual_costs,
                annual_benefits=low,
                discount_rate=base_calculator.discount_rate,
                time_horizon_years=base_calculator.time_horizon,
                adoption_curve=base_calculator.adoption_curve
            )
            calc_high = HealthcareAIROICalculator(
                year1_costs=base_calculator.year1_costs,
                ongoing_annual_costs=base_calculator.ongoing_annual_costs,
                annual_benefits=high,
                discount_rate=base_calculator.discount_rate,
                time_horizon_years=base_calculator.time_horizon,
                adoption_curve=base_calculator.adoption_curve
            )
        elif param == 'year1_costs':
            calc_low = HealthcareAIROICalculator(
                year1_costs=low,
                ongoing_annual_costs=base_calculator.ongoing_annual_costs,
                annual_benefits=base_calculator.annual_benefits,
                discount_rate=base_calculator.discount_rate,
                time_horizon_years=base_calculator.time_horizon,
                adoption_curve=base_calculator.adoption_curve
            )
            calc_high = HealthcareAIROICalculator(
                year1_costs=high,
                ongoing_annual_costs=base_calculator.ongoing_annual_costs,
                annual_benefits=base_calculator.annual_benefits,
                discount_rate=base_calculator.discount_rate,
                time_horizon_years=base_calculator.time_horizon,
                adoption_curve=base_calculator.adoption_curve
            )
        elif param == 'adoption_rate':
            low_adoption = [a * low for a in base_calculator.adoption_curve]
            high_adoption = [min(a * high, 1.0) for a in base_calculator.adoption_curve]
            calc_low = HealthcareAIROICalculator(
                year1_costs=base_calculator.year1_costs,
                ongoing_annual_costs=base_calculator.ongoing_annual_costs,
                annual_benefits=base_calculator.annual_benefits,
                discount_rate=base_calculator.discount_rate,
                time_horizon_years=base_calculator.time_horizon,
                adoption_curve=low_adoption
            )
            calc_high = HealthcareAIROICalculator(
                year1_costs=base_calculator.year1_costs,
                ongoing_annual_costs=base_calculator.ongoing_annual_costs,
                annual_benefits=base_calculator.annual_benefits,
                discount_rate=base_calculator.discount_rate,
                time_horizon_years=base_calculator.time_horizon,
                adoption_curve=high_adoption
            )
        elif param == 'ongoing_costs':
            calc_low = HealthcareAIROICalculator(
                year1_costs=base_calculator.year1_costs,
                ongoing_annual_costs=low,
                annual_benefits=base_calculator.annual_benefits,
                discount_rate=base_calculator.discount_rate,
                time_horizon_years=base_calculator.time_horizon,
                adoption_curve=base_calculator.adoption_curve
            )
            calc_high = HealthcareAIROICalculator(
                year1_costs=base_calculator.year1_costs,
                ongoing_annual_costs=high,
                annual_benefits=base_calculator.annual_benefits,
                discount_rate=base_calculator.discount_rate,
                time_horizon_years=base_calculator.time_horizon,
                adoption_curve=base_calculator.adoption_curve
            )
        else:
            continue
        
        npv_low = calc_low.calculate_npv()
        npv_high = calc_high.calculate_npv()
        
        results.append({
            'parameter': param,
            'low_value': low,
            'high_value': high,
            'npv_low': npv_low,
            'npv_high': npv_high,
            'npv_range': npv_high - npv_low,
            'impact_low': (npv_low - base_npv) / base_npv * 100,
            'impact_high': (npv_high - base_npv) / base_npv * 100
        })
    
    return pd.DataFrame(results)

# Define parameter ranges for sensitivity analysis
parameter_ranges = {
    'annual_benefits': (benefits['total_annual_benefit'] * 0.6, benefits['total_annual_benefit'] * 1.4),
    'year1_costs': (ai_scenario['year1_implementation_cost'] * 0.7, ai_scenario['year1_implementation_cost'] * 1.5),
    'adoption_rate': (0.6, 1.2),  # Multiplier for adoption curve
    'ongoing_costs': (ai_scenario['ongoing_annual_cost'] * 0.7, ai_scenario['ongoing_annual_cost'] * 1.5),
}

# Run sensitivity analysis
sensitivity_results = sensitivity_analysis(roi_calculator, parameter_ranges)

print("SENSITIVITY ANALYSIS RESULTS")
print("=" * 80)
print(f"\nBase Case NPV: {format_currency(roi_calculator.calculate_npv())}")
print("\nImpact of Parameter Changes on NPV:")
print("-" * 80)
for idx, row in sensitivity_results.iterrows():
    print(f"\n{row['parameter'].replace('_', ' ').title()}:")
    print(f"  Low scenario: NPV = {format_currency(row['npv_low'])} ({row['impact_low']:+.1f}%)")
    print(f"  High scenario: NPV = {format_currency(row['npv_high'])} ({row['impact_high']:+.1f}%)")

### Tornado Diagram

A tornado diagram visualizes which parameters have the greatest impact on the outcome.

In [ ]:
# Create tornado diagram
fig, ax = plt.subplots(figsize=(12, 6))

# Sort by impact range
sensitivity_results['abs_range'] = sensitivity_results['npv_range'].abs()
sensitivity_sorted = sensitivity_results.sort_values('abs_range', ascending=True)

base_npv = roi_calculator.calculate_npv()
y_pos = np.arange(len(sensitivity_sorted))

# Plot bars
labels = [p.replace('_', ' ').title() for p in sensitivity_sorted['parameter']]

# Calculate bar positions relative to base case
for i, (idx, row) in enumerate(sensitivity_sorted.iterrows()):
    low_delta = (row['npv_low'] - base_npv) / 1e6
    high_delta = (row['npv_high'] - base_npv) / 1e6
    
    # Plot low bar (red if negative impact)
    ax.barh(i, low_delta, height=0.6, color=COLORS['danger'] if low_delta < 0 else COLORS['success'],
            alpha=0.8, label='Low scenario' if i == 0 else '')
    # Plot high bar
    ax.barh(i, high_delta, height=0.6, color=COLORS['success'] if high_delta > 0 else COLORS['danger'],
            alpha=0.8, left=0, label='High scenario' if i == 0 else '')

ax.axvline(x=0, color='black', linestyle='-', linewidth=2)
ax.set_yticks(y_pos)
ax.set_yticklabels(labels, fontsize=11)
ax.set_xlabel('Change in NPV ($ Millions)', fontsize=12)
ax.set_title('Tornado Diagram: Sensitivity of NPV to Key Parameters', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

# Add annotation
ax.annotate(f'Base NPV: {format_currency(base_npv)}', xy=(0.02, 0.98), xycoords='axes fraction',
            fontsize=11, va='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

print("\nKey Insight: Parameters at the bottom of the tornado have the largest impact on NPV.")
print("Focus risk mitigation efforts on these high-impact variables.")

### Monte Carlo Simulation

Monte Carlo simulation provides probability distributions for outcomes by randomly sampling from parameter distributions.

In [ ]:
def monte_carlo_simulation(base_calculator, n_simulations=1000):
    """
    Run Monte Carlo simulation for ROI uncertainty analysis.
    
    Parameters are sampled from triangular distributions based on
    optimistic/pessimistic estimates.
    """
    np.random.seed(42)
    
    results = []
    
    for _ in range(n_simulations):
        # Sample parameters from distributions
        # Annual benefits: triangular (pessimistic, most likely, optimistic)
        benefits_mult = np.random.triangular(0.6, 1.0, 1.3)
        sampled_benefits = base_calculator.annual_benefits * benefits_mult
        
        # Year 1 costs: triangular (can be higher due to unforeseen issues)
        y1_cost_mult = np.random.triangular(0.9, 1.0, 1.5)
        sampled_y1_costs = base_calculator.year1_costs * y1_cost_mult
        
        # Ongoing costs: triangular
        ongoing_mult = np.random.triangular(0.85, 1.0, 1.3)
        sampled_ongoing = base_calculator.ongoing_annual_costs * ongoing_mult
        
        # Adoption rate: sample multiplier
        adoption_mult = np.random.triangular(0.7, 1.0, 1.1)
        sampled_adoption = [min(a * adoption_mult, 1.0) for a in base_calculator.adoption_curve]
        
        # Create calculator with sampled parameters
        sim_calc = HealthcareAIROICalculator(
            year1_costs=sampled_y1_costs,
            ongoing_annual_costs=sampled_ongoing,
            annual_benefits=sampled_benefits,
            discount_rate=base_calculator.discount_rate,
            time_horizon_years=base_calculator.time_horizon,
            adoption_curve=sampled_adoption
        )
        
        results.append({
            'npv': sim_calc.calculate_npv(),
            'roi': sim_calc.calculate_simple_roi(),
            'payback': sim_calc.calculate_payback_period(),
            'benefits_mult': benefits_mult,
            'y1_cost_mult': y1_cost_mult,
            'adoption_mult': adoption_mult
        })
    
    return pd.DataFrame(results)

# Run Monte Carlo simulation
mc_results = monte_carlo_simulation(roi_calculator, n_simulations=5000)

# Calculate confidence intervals
npv_percentiles = np.percentile(mc_results['npv'], [5, 25, 50, 75, 95])
roi_percentiles = np.percentile(mc_results['roi'], [5, 25, 50, 75, 95])

print("MONTE CARLO SIMULATION RESULTS (5,000 iterations)")
print("=" * 60)
print(f"\nNPV Distribution:")
print(f"  5th percentile (pessimistic): {format_currency(npv_percentiles[0])}")
print(f"  25th percentile: {format_currency(npv_percentiles[1])}")
print(f"  50th percentile (median): {format_currency(npv_percentiles[2])}")
print(f"  75th percentile: {format_currency(npv_percentiles[3])}")
print(f"  95th percentile (optimistic): {format_currency(npv_percentiles[4])}")
print(f"\nProbability of positive NPV: {(mc_results['npv'] > 0).mean():.1%}")
print(f"Probability of NPV > $1M: {(mc_results['npv'] > 1e6).mean():.1%}")
print(f"\nROI Distribution:")
print(f"  5th percentile: {roi_percentiles[0]:.1%}")
print(f"  Median: {roi_percentiles[2]:.1%}")
print(f"  95th percentile: {roi_percentiles[4]:.1%}")

In [ ]:
# Visualize Monte Carlo results
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# NPV distribution
ax = axes[0]
ax.hist(mc_results['npv']/1e6, bins=50, color=COLORS['primary'], alpha=0.7, edgecolor='black')
ax.axvline(x=0, color=COLORS['danger'], linestyle='--', linewidth=2, label='Break-even')
ax.axvline(x=np.median(mc_results['npv'])/1e6, color=COLORS['success'], linestyle='-', 
           linewidth=2, label=f'Median: {format_currency(np.median(mc_results["npv"]))}')
ax.set_xlabel('NPV ($ Millions)', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title('NPV Distribution (Monte Carlo)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# ROI distribution
ax = axes[1]
ax.hist(mc_results['roi']*100, bins=50, color=COLORS['success'], alpha=0.7, edgecolor='black')
ax.axvline(x=0, color=COLORS['danger'], linestyle='--', linewidth=2, label='Break-even')
ax.axvline(x=np.median(mc_results['roi'])*100, color=COLORS['primary'], linestyle='-', 
           linewidth=2, label=f'Median: {np.median(mc_results["roi"]):.1%}')
ax.set_xlabel('ROI (%)', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title('ROI Distribution (Monte Carlo)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Payback period distribution
ax = axes[2]
valid_payback = mc_results['payback'].dropna()
ax.hist(valid_payback, bins=30, color=COLORS['warning'], alpha=0.7, edgecolor='black')
ax.axvline(x=np.median(valid_payback), color=COLORS['primary'], linestyle='-', 
           linewidth=2, label=f'Median: {np.median(valid_payback):.1f} months')
ax.set_xlabel('Payback Period (months)', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title('Payback Period Distribution', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Break-even Analysis: What Adoption Rate is Needed?

In [ ]:
# Analyze break-even at different adoption rates
adoption_rates = np.arange(0.1, 1.05, 0.05)
break_even_analysis = []

for adoption in adoption_rates:
    adoption_curve = [adoption] * 5  # Constant adoption
    calc = HealthcareAIROICalculator(
        year1_costs=ai_scenario['year1_implementation_cost'],
        ongoing_annual_costs=ai_scenario['ongoing_annual_cost'],
        annual_benefits=benefits['total_annual_benefit'],
        discount_rate=0.08,
        time_horizon_years=5,
        adoption_curve=adoption_curve
    )
    
    break_even_analysis.append({
        'adoption_rate': adoption,
        'npv': calc.calculate_npv(),
        'roi': calc.calculate_simple_roi(),
        'payback': calc.calculate_payback_period()
    })

df_breakeven = pd.DataFrame(break_even_analysis)

# Find break-even adoption rate
break_even_adoption = df_breakeven[df_breakeven['npv'] >= 0]['adoption_rate'].min()

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# NPV vs adoption rate
ax = axes[0]
ax.plot(df_breakeven['adoption_rate']*100, df_breakeven['npv']/1e6, 'o-', 
        color=COLORS['primary'], linewidth=2, markersize=8)
ax.axhline(y=0, color=COLORS['danger'], linestyle='--', linewidth=2, label='Break-even NPV')
ax.axvline(x=break_even_adoption*100, color=COLORS['warning'], linestyle=':', 
           linewidth=2, label=f'Break-even adoption: {break_even_adoption:.0%}')
ax.fill_between(df_breakeven['adoption_rate']*100, df_breakeven['npv']/1e6, 0,
                where=df_breakeven['npv'] >= 0, alpha=0.3, color=COLORS['success'])
ax.fill_between(df_breakeven['adoption_rate']*100, df_breakeven['npv']/1e6, 0,
                where=df_breakeven['npv'] < 0, alpha=0.3, color=COLORS['danger'])
ax.set_xlabel('Adoption Rate (%)', fontsize=12)
ax.set_ylabel('NPV ($ Millions)', fontsize=12)
ax.set_title('Break-even Analysis: NPV vs. Adoption Rate', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# ROI vs adoption rate
ax = axes[1]
ax.plot(df_breakeven['adoption_rate']*100, df_breakeven['roi']*100, 's-', 
        color=COLORS['success'], linewidth=2, markersize=8)
ax.axhline(y=0, color=COLORS['danger'], linestyle='--', linewidth=2)
ax.axhline(y=20, color=COLORS['warning'], linestyle=':', linewidth=2, label='20% ROI target')
ax.set_xlabel('Adoption Rate (%)', fontsize=12)
ax.set_ylabel('ROI (%)', fontsize=12)
ax.set_title('ROI vs. Adoption Rate', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nMinimum adoption rate for break-even: {break_even_adoption:.0%}")
print(f"Adoption rate for 20% ROI: {df_breakeven[df_breakeven['roi'] >= 0.20]['adoption_rate'].min():.0%}")

---

## 5. Time-to-Value Analysis

The "J-curve" of AI implementation shows that value often dips before rising, as implementation costs precede benefit realization.

In [ ]:
def calculate_monthly_value(calculator, months=60):
    """
    Calculate monthly cumulative value over implementation timeline.
    """
    # Implementation phases (months)
    phases = {
        'planning': (0, 3),       # Months 1-3
        'integration': (3, 6),     # Months 4-6
        'pilot': (6, 12),          # Months 7-12
        'rollout': (12, 18),       # Months 13-18
        'optimization': (18, 24),  # Months 19-24
        'steady_state': (24, months)  # Month 25+
    }
    
    # Cost distribution by phase
    monthly_data = []
    cumulative_cost = 0
    cumulative_benefit = 0
    
    for month in range(1, months + 1):
        # Determine phase
        for phase, (start, end) in phases.items():
            if start < month <= end:
                current_phase = phase
                break
        
        # Calculate monthly costs
        if month <= 3:  # Planning
            monthly_cost = calculator.year1_costs * 0.15 / 3
            benefit_realization = 0
        elif month <= 6:  # Integration
            monthly_cost = calculator.year1_costs * 0.45 / 3
            benefit_realization = 0
        elif month <= 12:  # Pilot
            monthly_cost = calculator.year1_costs * 0.30 / 6
            benefit_realization = 0.10  # 10% of potential benefit
        elif month <= 18:  # Rollout
            monthly_cost = calculator.year1_costs * 0.10 / 6 + calculator.ongoing_annual_costs / 12
            benefit_realization = 0.35
        elif month <= 24:  # Optimization
            monthly_cost = calculator.ongoing_annual_costs / 12
            benefit_realization = 0.65
        else:  # Steady state
            monthly_cost = calculator.ongoing_annual_costs / 12
            benefit_realization = min(0.65 + (month - 24) * 0.02, 1.0)  # Gradual increase to 100%
        
        monthly_benefit = (calculator.annual_benefits / 12) * benefit_realization
        
        cumulative_cost += monthly_cost
        cumulative_benefit += monthly_benefit
        
        monthly_data.append({
            'month': month,
            'phase': current_phase,
            'monthly_cost': monthly_cost,
            'monthly_benefit': monthly_benefit,
            'net_monthly': monthly_benefit - monthly_cost,
            'cumulative_cost': cumulative_cost,
            'cumulative_benefit': cumulative_benefit,
            'cumulative_net': cumulative_benefit - cumulative_cost,
            'benefit_realization': benefit_realization
        })
    
    return pd.DataFrame(monthly_data)

# Calculate monthly value
monthly_value = calculate_monthly_value(roi_calculator, months=48)

# Find time to positive cumulative value
time_to_value = monthly_value[monthly_value['cumulative_net'] >= 0]['month'].min()

print(f"TIME-TO-VALUE ANALYSIS")
print("=" * 60)
print(f"\nTime to positive cumulative value: Month {time_to_value}")
print(f"Maximum negative cumulative value: {format_currency(monthly_value['cumulative_net'].min())}")
print(f"Cumulative value at Month 24: {format_currency(monthly_value[monthly_value['month']==24]['cumulative_net'].values[0])}")
print(f"Cumulative value at Month 48: {format_currency(monthly_value[monthly_value['month']==48]['cumulative_net'].values[0])}")

In [ ]:
# Visualize the J-curve
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# J-curve: Cumulative net value
ax = axes[0, 0]
cumulative_net = monthly_value['cumulative_net'] / 1e6
colors = [COLORS['danger'] if v < 0 else COLORS['success'] for v in cumulative_net]
ax.fill_between(monthly_value['month'], cumulative_net, 0, 
                where=cumulative_net >= 0, alpha=0.3, color=COLORS['success'])
ax.fill_between(monthly_value['month'], cumulative_net, 0, 
                where=cumulative_net < 0, alpha=0.3, color=COLORS['danger'])
ax.plot(monthly_value['month'], cumulative_net, color=COLORS['primary'], linewidth=2)
ax.axhline(y=0, color='black', linestyle='-', linewidth=1)
ax.axvline(x=time_to_value, color=COLORS['warning'], linestyle='--', linewidth=2,
           label=f'Time to Value: Month {time_to_value}')
ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Cumulative Net Value ($ Millions)', fontsize=12)
ax.set_title('The J-Curve: AI Implementation Value Over Time', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Add phase annotations
phase_boundaries = [0, 3, 6, 12, 18, 24]
phase_labels = ['Plan', 'Build', 'Pilot', 'Rollout', 'Optimize']
for i, (start, label) in enumerate(zip(phase_boundaries[:-1], phase_labels)):
    mid = (phase_boundaries[i] + phase_boundaries[i+1]) / 2
    ax.annotate(label, xy=(mid, ax.get_ylim()[0]*0.9), ha='center', fontsize=9, style='italic')

# Cumulative costs vs benefits
ax = axes[0, 1]
ax.plot(monthly_value['month'], monthly_value['cumulative_cost']/1e6, 
        label='Cumulative Costs', color=COLORS['danger'], linewidth=2)
ax.plot(monthly_value['month'], monthly_value['cumulative_benefit']/1e6, 
        label='Cumulative Benefits', color=COLORS['success'], linewidth=2)
ax.axvline(x=time_to_value, color=COLORS['warning'], linestyle='--', linewidth=2,
           label='Break-even')
ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Cumulative Amount ($ Millions)', fontsize=12)
ax.set_title('Cumulative Costs vs. Benefits', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Benefit realization over time
ax = axes[1, 0]
ax.fill_between(monthly_value['month'], monthly_value['benefit_realization']*100,
                alpha=0.5, color=COLORS['primary'])
ax.plot(monthly_value['month'], monthly_value['benefit_realization']*100, 
        color=COLORS['primary'], linewidth=2)
ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Benefit Realization (%)', fontsize=12)
ax.set_title('Benefit Realization Curve', fontsize=13, fontweight='bold')
ax.set_ylim(0, 105)
ax.grid(True, alpha=0.3)

# Monthly cash flow
ax = axes[1, 1]
ax.bar(monthly_value['month'], monthly_value['net_monthly']/1e3,
       color=[COLORS['danger'] if v < 0 else COLORS['success'] for v in monthly_value['net_monthly']],
       alpha=0.7)
ax.axhline(y=0, color='black', linestyle='-', linewidth=1)
ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Monthly Net Cash Flow ($K)', fontsize=12)
ax.set_title('Monthly Net Cash Flow', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

---

## 6. Success Dashboard Design

A well-designed dashboard helps stakeholders track progress and make timely decisions.

In [ ]:
def generate_dashboard_data(months=12):
    """
    Generate simulated dashboard data for demonstration.
    """
    np.random.seed(42)
    
    data = []
    for month in range(1, months + 1):
        # Simulated metrics with realistic variance
        adoption = min(0.15 + month * 0.07 + np.random.normal(0, 0.03), 1.0)
        alerts_generated = int(45 * adoption * (1 + np.random.normal(0, 0.1)))
        alerts_acted_on = int(alerts_generated * (0.6 + month * 0.02))
        true_positives = int(alerts_acted_on * 0.75)
        lives_saved = int(2.5 * adoption * (1 + np.random.normal(0, 0.2)))
        los_reduction = 1.5 + month * 0.05 + np.random.normal(0, 0.2)
        cost_savings = 150000 * adoption * (1 + np.random.normal(0, 0.1))
        
        # Targets
        adoption_target = 0.15 + month * 0.07
        savings_target = 150000 * adoption_target
        
        data.append({
            'month': month,
            'adoption_rate': adoption,
            'adoption_target': adoption_target,
            'alerts_generated': alerts_generated,
            'alerts_acted_on': alerts_acted_on,
            'true_positives': true_positives,
            'ppv': true_positives / alerts_acted_on if alerts_acted_on > 0 else 0,
            'lives_saved': lives_saved,
            'cumulative_lives_saved': 0,  # Will calculate
            'los_reduction': los_reduction,
            'cost_savings': cost_savings,
            'cost_savings_target': savings_target,
            'cumulative_savings': 0  # Will calculate
        })
    
    df = pd.DataFrame(data)
    df['cumulative_lives_saved'] = df['lives_saved'].cumsum()
    df['cumulative_savings'] = df['cost_savings'].cumsum()
    
    return df

# Generate dashboard data
dashboard_data = generate_dashboard_data(12)

def status_indicator(actual, target, threshold=0.1):
    """Return traffic light status based on target achievement."""
    ratio = actual / target if target > 0 else 0
    if ratio >= 1.0:
        return 'green', 'On Track'
    elif ratio >= (1 - threshold):
        return 'yellow', 'At Risk'
    else:
        return 'red', 'Behind'

# Current month metrics
current = dashboard_data.iloc[-1]

print("SUCCESS TRACKING DASHBOARD - Month 12")
print("=" * 70)

# Key metrics with status
metrics = [
    ('Adoption Rate', current['adoption_rate'], current['adoption_target'], '%'),
    ('Monthly Cost Savings', current['cost_savings'], current['cost_savings_target'], '$'),
    ('Positive Predictive Value', current['ppv'], 0.70, '%'),
    ('LOS Reduction (days)', current['los_reduction'], 2.0, 'days'),
]

print(f"\n{'Metric':<30} {'Actual':<15} {'Target':<15} {'Status':<15}")
print("-" * 70)

for name, actual, target, unit in metrics:
    color, status = status_indicator(actual, target)
    if unit == '%':
        actual_str = f"{actual:.1%}"
        target_str = f"{target:.1%}"
    elif unit == '$':
        actual_str = format_currency(actual)
        target_str = format_currency(target)
    else:
        actual_str = f"{actual:.1f} {unit}"
        target_str = f"{target:.1f} {unit}"
    
    status_symbol = {'green': '[OK]', 'yellow': '[!!]', 'red': '[XX]'}[color]
    print(f"{name:<30} {actual_str:<15} {target_str:<15} {status_symbol} {status}")

print(f"\nCumulative Performance:")
print(f"  Total Lives Saved: {current['cumulative_lives_saved']:.0f}")
print(f"  Total Cost Savings: {format_currency(current['cumulative_savings'])}")

In [ ]:
# Visual dashboard
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# 1. Adoption rate vs target
ax = axes[0, 0]
ax.plot(dashboard_data['month'], dashboard_data['adoption_rate']*100, 'o-', 
        color=COLORS['primary'], linewidth=2, markersize=8, label='Actual')
ax.plot(dashboard_data['month'], dashboard_data['adoption_target']*100, '--', 
        color=COLORS['neutral'], linewidth=2, label='Target')
ax.fill_between(dashboard_data['month'], dashboard_data['adoption_rate']*100, 
                dashboard_data['adoption_target']*100,
                where=dashboard_data['adoption_rate'] >= dashboard_data['adoption_target'],
                alpha=0.3, color=COLORS['success'])
ax.fill_between(dashboard_data['month'], dashboard_data['adoption_rate']*100, 
                dashboard_data['adoption_target']*100,
                where=dashboard_data['adoption_rate'] < dashboard_data['adoption_target'],
                alpha=0.3, color=COLORS['danger'])
ax.set_xlabel('Month', fontsize=11)
ax.set_ylabel('Adoption Rate (%)', fontsize=11)
ax.set_title('Adoption Rate vs Target', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# 2. Monthly cost savings
ax = axes[0, 1]
ax.bar(dashboard_data['month'], dashboard_data['cost_savings']/1e3, 
       color=COLORS['success'], alpha=0.8, label='Actual')
ax.plot(dashboard_data['month'], dashboard_data['cost_savings_target']/1e3, 'o--', 
        color=COLORS['danger'], linewidth=2, markersize=6, label='Target')
ax.set_xlabel('Month', fontsize=11)
ax.set_ylabel('Monthly Savings ($K)', fontsize=11)
ax.set_title('Monthly Cost Savings', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

# 3. Cumulative lives saved
ax = axes[0, 2]
ax.fill_between(dashboard_data['month'], dashboard_data['cumulative_lives_saved'],
                alpha=0.5, color=COLORS['primary'])
ax.plot(dashboard_data['month'], dashboard_data['cumulative_lives_saved'], 'o-',
        color=COLORS['primary'], linewidth=2, markersize=8)
ax.set_xlabel('Month', fontsize=11)
ax.set_ylabel('Cumulative Lives Saved', fontsize=11)
ax.set_title('Cumulative Clinical Impact', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)

# 4. PPV trend
ax = axes[1, 0]
ax.plot(dashboard_data['month'], dashboard_data['ppv']*100, 'o-',
        color=COLORS['warning'], linewidth=2, markersize=8)
ax.axhline(y=70, color=COLORS['success'], linestyle='--', linewidth=2, label='Target (70%)')
ax.set_xlabel('Month', fontsize=11)
ax.set_ylabel('Positive Predictive Value (%)', fontsize=11)
ax.set_title('Alert Quality (PPV)', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(50, 100)
ax.grid(True, alpha=0.3)

# 5. LOS reduction trend
ax = axes[1, 1]
ax.bar(dashboard_data['month'], dashboard_data['los_reduction'],
       color=COLORS['primary'], alpha=0.8)
ax.axhline(y=2.0, color=COLORS['success'], linestyle='--', linewidth=2, label='Target (2 days)')
ax.set_xlabel('Month', fontsize=11)
ax.set_ylabel('LOS Reduction (days)', fontsize=11)
ax.set_title('Length of Stay Reduction', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

# 6. Cumulative savings vs investment
ax = axes[1, 2]
investment = ai_scenario['year1_implementation_cost']  # Year 1 investment
ax.fill_between(dashboard_data['month'], dashboard_data['cumulative_savings']/1e6,
                alpha=0.5, color=COLORS['success'], label='Cumulative Savings')
ax.axhline(y=investment/1e6, color=COLORS['danger'], linestyle='--', 
           linewidth=2, label=f'Year 1 Investment ({format_currency(investment)})')
ax.set_xlabel('Month', fontsize=11)
ax.set_ylabel('Amount ($ Millions)', fontsize=11)
ax.set_title('Cumulative Savings vs Investment', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('success_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nDashboard saved as 'success_dashboard.png'")

---

## 7. Common Pitfalls in AI ROI Calculations

ROI analyses can be misleading if common pitfalls are not addressed.

In [ ]:
# Demonstrate impact of common pitfalls

pitfalls = {
    'Overestimating Adoption': {
        'description': 'Assuming 100% adoption from day one',
        'realistic_adoption': [0.30, 0.65, 0.85, 0.95, 1.0],
        'optimistic_adoption': [0.80, 0.95, 1.0, 1.0, 1.0],
    },
    'Underestimating Change Management': {
        'description': 'Forgetting training and workflow redesign costs',
        'realistic_cost_multiplier': 1.0,
        'optimistic_cost_multiplier': 0.7,
    },
    'Ignoring Maintenance Costs': {
        'description': 'Not budgeting for model retraining and updates',
        'realistic_annual_maintenance': 0.20,  # 20% of software cost
        'optimistic_annual_maintenance': 0.05,  # Only 5%
    },
    'Attribution Challenges': {
        'description': 'Attributing all improvement to AI (ignoring confounders)',
        'realistic_attribution': 0.70,  # 70% of observed benefit is AI
        'optimistic_attribution': 1.0,  # 100% attribution
    }
}

# Calculate ROI under realistic vs optimistic assumptions
results_comparison = []

# Realistic scenario (base case)
realistic_calc = roi_calculator
realistic_npv = realistic_calc.calculate_npv()
realistic_roi = realistic_calc.calculate_simple_roi()

# Optimistic scenario (common pitfalls)
optimistic_calc = HealthcareAIROICalculator(
    year1_costs=ai_scenario['year1_implementation_cost'] * 0.7,  # Underestimate costs
    ongoing_annual_costs=ai_scenario['ongoing_annual_cost'] * 0.6,  # Ignore maintenance
    annual_benefits=benefits['total_annual_benefit'] * 1.3,  # Over-attribute benefits
    discount_rate=0.08,
    time_horizon_years=5,
    adoption_curve=[0.80, 0.95, 1.0, 1.0, 1.0]  # Overestimate adoption
)
optimistic_npv = optimistic_calc.calculate_npv()
optimistic_roi = optimistic_calc.calculate_simple_roi()

print("IMPACT OF COMMON PITFALLS ON ROI")
print("=" * 70)
print(f"\n{'Scenario':<25} {'NPV':<20} {'ROI':<15} {'Payback':<15}")
print("-" * 70)
print(f"{'Realistic (recommended)':<25} {format_currency(realistic_npv):<20} {realistic_roi:<15.1%} {realistic_calc.calculate_payback_period():<15.1f} mo")
print(f"{'Optimistic (pitfalls)':<25} {format_currency(optimistic_npv):<20} {optimistic_roi:<15.1%} {optimistic_calc.calculate_payback_period():<15.1f} mo")
print(f"\nDifference: NPV is {(optimistic_npv - realistic_npv)/realistic_npv:.0%} higher with optimistic assumptions!")

print("\n" + "=" * 70)
print("COMMON PITFALLS AND HOW TO AVOID THEM")
print("=" * 70)

pitfall_details = [
    ('1. Overestimating Adoption Rate', 
     'Use phased adoption curves based on similar implementations',
     'Realistic: 30% Year 1 -> 100% Year 5'),
    ('2. Underestimating Change Management',
     'Budget for training, workflow redesign, and clinician time',
     'Add 15-25% to implementation costs'),
    ('3. Ignoring Maintenance Costs',
     'Include model retraining, updates, and IT support',
     'Budget 15-25% of software cost annually'),
    ('4. Attribution Challenges',
     'Use control groups or difference-in-differences analysis',
     'Discount observed benefits by 20-40%'),
    ('5. Ignoring Opportunity Costs',
     'Account for staff time diverted to implementation',
     'Include clinical time for training and adaptation'),
]

for pitfall, solution, example in pitfall_details:
    print(f"\n{pitfall}")
    print(f"  Solution: {solution}")
    print(f"  Example: {example}")

In [ ]:
# Visualize realistic vs optimistic projections
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Cash flows comparison
ax = axes[0]
realistic_cf = realistic_calc.calculate_cash_flows()
optimistic_cf = optimistic_calc.calculate_cash_flows()

x = realistic_cf['year']
ax.plot(x, realistic_cf['net_cash_flow'].cumsum()/1e6, 'o-', 
        color=COLORS['primary'], linewidth=2, markersize=8, label='Realistic')
ax.plot(x, optimistic_cf['net_cash_flow'].cumsum()/1e6, 's--', 
        color=COLORS['warning'], linewidth=2, markersize=8, label='Optimistic (pitfalls)')
ax.axhline(y=0, color='black', linestyle='-', linewidth=1)
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Cumulative Net Value ($ Millions)', fontsize=12)
ax.set_title('Cumulative Value: Realistic vs Optimistic', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# Gap between scenarios
ax = axes[1]
gap = (optimistic_cf['net_cash_flow'].cumsum() - realistic_cf['net_cash_flow'].cumsum()) / 1e6
ax.bar(x, gap, color=COLORS['danger'], alpha=0.8)
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Optimism Gap ($ Millions)', fontsize=12)
ax.set_title('The "Optimism Gap": Overestimation by Year', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Add annotation
total_gap = gap.iloc[-1]
ax.annotate(f'Total gap: {format_currency(total_gap*1e6)}', 
            xy=(0.7, 0.9), xycoords='axes fraction',
            fontsize=12, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

---

## 8. Summary and Best Practices

### Key Takeaways

1. **Define metrics BEFORE implementation**: Success criteria must be established upfront, not retroactively.

2. **Use multiple metric categories**: Clinical outcomes, operational efficiency, and financial performance all matter.

3. **Build realistic adoption curves**: Expect a J-curve with initial costs before benefits materialize.

4. **Perform sensitivity analysis**: Understand which assumptions have the greatest impact on ROI.

5. **Track progress against targets**: Regular dashboard reviews enable course corrections.

6. **Avoid common pitfalls**: Be conservative on adoption rates, comprehensive on costs, and honest about attribution.

In [ ]:
# Summary table of best practices
best_practices = pd.DataFrame({
    'Category': ['Metric Selection', 'Metric Selection', 'Metric Selection',
                 'ROI Calculation', 'ROI Calculation', 'ROI Calculation',
                 'Timeline', 'Timeline', 'Timeline',
                 'Tracking', 'Tracking', 'Tracking'],
    'Best Practice': [
        'Include clinical, operational, AND financial metrics',
        'Define baseline before implementation',
        'Establish clear targets with timelines',
        'Use phased adoption curves (not immediate 100%)',
        'Include ALL costs (change management, maintenance)',
        'Discount benefits for attribution uncertainty',
        'Expect 18-24 months to positive cumulative value',
        'Plan for pilot (3-6 mo) before full rollout',
        'Budget for 5-year time horizon minimum',
        'Review dashboard monthly with stakeholders',
        'Use traffic light indicators for quick status',
        'Document and explain variances from targets'
    ],
    'Common Mistake': [
        'Only tracking financial metrics',
        'No pre-implementation baseline',
        'Vague goals without numbers',
        'Assuming immediate full adoption',
        'Forgetting training and support costs',
        'Attributing 100% of improvement to AI',
        'Expecting ROI in 6 months',
        'Skipping pilot phase',
        'Only analyzing 1-2 years',
        'Annual reviews only',
        'Complex dashboards nobody reads',
        'Ignoring negative variance'
    ]
})

print("\nBEST PRACTICES SUMMARY")
print("=" * 100)
print(best_practices.to_string(index=False))

In [ ]:
# ROI Calculation Checklist
print("\nROI CALCULATION CHECKLIST")
print("=" * 60)

checklist = [
    ("COSTS", [
        "Software licensing (annual)",
        "Integration and customization (one-time)",
        "Hardware/infrastructure (if needed)",
        "Initial training program",
        "Change management and communication",
        "Ongoing maintenance (15-25% of software)",
        "Ongoing training (refreshers, new staff)",
        "IT support allocation",
        "Clinical time for adaptation"
    ]),
    ("BENEFITS", [
        "Mortality cost avoided (with attribution discount)",
        "Length of stay reduction",
        "ICU days avoided",
        "Readmission reduction",
        "Complication avoidance",
        "Time savings (clinician hours)",
        "Throughput improvement",
        "Revenue impact (if applicable)"
    ]),
    ("ASSUMPTIONS TO DOCUMENT", [
        "Adoption curve by month/year",
        "Discount rate for NPV",
        "Attribution percentage (% of benefit due to AI)",
        "Time horizon for analysis",
        "Sensitivity ranges for key parameters"
    ])
]

for category, items in checklist:
    print(f"\n{category}:")
    for item in items:
        print(f"  [ ] {item}")

In [ ]:
# Final summary
print("\n" + "=" * 70)
print("NOTEBOOK COMPLETE: Success Metrics and ROI for Healthcare AI")
print("=" * 70)
print("""
Key concepts covered:

  [1] Three-dimensional success metrics (Clinical, Operational, Financial)
  [2] Comprehensive ROI model with NPV, payback period, and break-even analysis
  [3] Sensitivity analysis with tornado diagrams
  [4] Monte Carlo simulation for uncertainty quantification
  [5] Time-to-value analysis and the J-curve
  [6] Success dashboard design with traffic light indicators
  [7] Common pitfalls and how to avoid them

Remember: The goal is not to make AI look good on paper,
but to make honest assessments that lead to good decisions.
""")

---

## Exercises

1. **Modify the baseline scenario**: Change the annual sepsis cases from 500 to 800 and recalculate the ROI. How does scale affect the payback period?

2. **Create a pessimistic scenario**: Build an ROI model where adoption reaches only 50% by Year 5. What is the minimum adoption rate needed for break-even?

3. **Add a new benefit category**: Extend the benefits calculation to include reduction in nursing overtime. Assume each sepsis case requires 4 hours of overtime at $75/hour, and AI reduces this by 30%.

4. **Design your own dashboard**: Create a dashboard for a different AI use case (e.g., radiology AI for chest X-ray interpretation). What metrics would you track?

5. **Sensitivity analysis deep dive**: Identify the break-even point for each key parameter. At what value does each parameter cause the NPV to go negative?

6. **Real-world calibration**: Research published ROI studies for healthcare AI (sepsis prediction, radiology AI, etc.) and compare their assumptions to this model. Are the assumptions used here conservative or optimistic?

---

*This notebook accompanies Chapter 6 of AI in Healthcare Volume 2. For the full strategic and organizational context, please refer to the textbook.*